In [1]:
# import required packages
import os
import pandas as pd
import numpy as np

# JR6 data conversion

In [3]:
path = r'G:\My Drive\__Postdoc\Data\FSU\Panther_Creek' #os.getcwd()
file = 'Panther_Creek_IZZI_test.jr6'
volume = 11 #cc

protocol = 'IZZI' # 'PTSD' 'Dir'
# for pseuro-Thellier and Shaw experiments: ARM demag series have to have .01 as identifier in AF field

DC_field = 48.3 # uT value for the used lab field in intensity experiments
DC_phi = 0 # 'declination' of applied field 
DC_theta = -90 # 'inclination' of applied field 

site_delimiter = '' # e.g., '-' for MAT1-1.2 ([site]-[sample].[specimen]); if there is none, use ''
site_chars = 2 # number of characters at the beginning of the sample name that are the site identifier. only used if 
               # site_delimiter = ''
specimen_delimiter = '-' # e.g., '.' for MAT1-1.2 ([site]-[sample].[specimen]); if specimen = sample, use ''
specimen_chars = 1 # number of characters from the end of the sample names if there is no delimiter (use 0 for spec = sample)

#---------------------------------------------------------------------------------------------


# Start new Magic contribution for a locality

sites_dict = {'site': [], 'method_codes': [], 'citations': []}
samples_dict = {'site': [], 'sample': [],'method_codes': [], 'citations': [],
                'azimuth': [], 'dip': [], 'bed_dip_direction': [], 'bed_dip': [],'result_quality': []}
specimens_dict = {'specimen': [], 'sample': [],'method_codes': [],'experiments': [],'volume': [], 'citations': [],
                  'result_quality':[]}
measurement_dict = {'specimen': [],'method_codes': [],'dir_dec': [],'dir_inc': [],'magn_volume': [], 'experiment': [],
                    'dir_csd': [],'treat_temp':[], 'meas_temp': [],'treat_ac_field': [], 'treat_dc_field': [],'citations': [],
                     'treat_dc_field_phi': [],'instrument_codes': [],'sequence':[], 'magn_moment': [], 'measurement': [],
                    'treat_dc_field_theta': [],'treat_step_num':[],'quality': []}

if not path.endswith('/'): path += '/'
jr6_raw = pd.read_fwf(path+file,widths=[10,8,6,6,6,4,4,4,4,4,4,4,3,3,3,3,4],header=None,names=['specimen','treat','m_x','m_y','m_z','exp','azimuth','spec_plunge',
                                                            'bed_dip_direction','lin_trend','lin_plunge','bed_plunge','orient_1',
                                                            'orient_2','orient_3','orient_4','csd'])
jr6_raw['m_x'] = (jr6_raw['m_x'].astype(str)+'E'+jr6_raw['exp'].astype(str)).astype(float)
jr6_raw['m_y'] = (jr6_raw['m_y'].astype(str)+'E'+jr6_raw['exp'].astype(str)).astype(float)
jr6_raw['m_z'] = (jr6_raw['m_z'].astype(str)+'E'+jr6_raw['exp'].astype(str)).astype(float)
jr6_raw['magn_volume'] = np.sqrt(jr6_raw['m_x']**2 + jr6_raw['m_y']**2 + jr6_raw['m_z']**2) # A/m
jr6_raw['magn_moment'] = jr6_raw['magn_volume']*(float(volume)/10**6) # Am2
jr6_raw['dir_dec'] = round((np.arctan2(jr6_raw['m_y'],jr6_raw['m_x'])*180/np.pi)%360,2)
jr6_raw['dir_inc'] = round((np.arctan2(jr6_raw['m_z'],np.sqrt(jr6_raw['m_x']**2+jr6_raw['m_y']**2)) *180/np.pi),2) # A/m

#changing orientations based on AGICO parameters (someone should probably check this):
# nothing to be done for P1(?)
#jr6_raw.loc[jr6_raw['orient_1'] < 12]['azimuth'] = jr6_raw.loc[jr6_raw['orient_1'] < 12]['azimuth'] - (jr6_raw.loc[jr6_raw['orient_1'] < 12]['orient_1'] / 12 * 60)
jr6_raw.loc[jr6_raw['orient_2'] == 0,'spec_plunge'] = 90 - jr6_raw.loc[jr6_raw['orient_2'] == 0,'spec_plunge'] #(P2==0 -> cube)
jr6_raw.loc[jr6_raw['orient_3'] < 12,'azimuth'] = jr6_raw.loc[jr6_raw['orient_3'] < 12,'azimuth'] - (jr6_raw.loc[jr6_raw['orient_3'] < 12,'orient_3'] / 12 * 60)
jr6_raw['bed_dip_direction'] = jr6_raw['bed_dip_direction'] + jr6_raw['orient_4']

# currently, lineation information is not used for anything - no clue what it is called in MagIC
#if not sum(jr6_raw['lin_trend']) == 0 and not sum(jr6_raw['lin_plunge']) == 0: 
#    samples_dict['lin_trend'] = []
#    samples_dict['lin_plunge'] = []
jr6_raw['dir_csd'] = np.zeros(len(jr6_raw))
jr6_raw['meas_temp'] = np.ones(len(jr6_raw))*273
jr6_raw['volume'] = np.ones(len(jr6_raw))*float(volume)/10**6
jr6_raw['dip'] = jr6_raw['spec_plunge']-90
jr6_raw['instrument_codes'] = 'JR6'
jr6_raw['quality'] = 'g'
jr6_raw['result_quality'] = 'g'
jr6_raw['citations'] = 'This study'
jr6_raw['bed_dip'] = jr6_raw['bed_plunge']#-90
jr6_raw['treat_dc_field'] = np.zeros(len(jr6_raw))#np.ones(len(jr6_raw)) * DC_field *10**-6
jr6_raw['treat_dc_field_phi'] = np.zeros(len(jr6_raw))#np.ones(len(jr6_raw)) * DC_phi
jr6_raw['treat_dc_field_theta'] = np.zeros(len(jr6_raw))#np.ones(len(jr6_raw)) * DC_theta

jr6_raw['index'] = jr6_raw.index
jr6_raw = jr6_raw.sort_values(by=['specimen','index'])
site, sample, method_code, treat_ac_field, experiment,treat_temp, measurement = [],[],[],[],[],[],[]
sequence, treat_step_num = [],[]
if not protocol == 'Dir': treat_dc_field, treat_dc_field_phi, treat_dc_field_theta = [],[],[]
lastname = ''
for i in range(len(jr6_raw)):
    if i == 0: lastname = jr6_raw.loc[jr6_raw.index[i],'specimen']
    if not site_delimiter == '': 
        site.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen']).split(site_delimiter)[0])
    else: 
        site.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])[:site_chars+1])
                       
    if not specimen_delimiter == '': sample.append(''.join(jr6_raw.loc[jr6_raw.index[i],'specimen'].split(specimen_delimiter)[:-1]))
    else:
        if not specimen_chars == 0:
            sample.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])[:-specimen_chars])
        else:
            sample.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen']))
            
    if i > 0 and lastname == jr6_raw.loc[jr6_raw.index[i],'specimen']:
        sequence.append(int(sequence[-1])+1)
    else:
        sequence.append(0)
    if sequence[-1] < 10: seq_str = '00'+str(sequence[-1])
    elif i < 100: seq_str = '0'+str(sequence[-1])
    else: seq_str = str(sequence[-1])
    treat_step_num.append(int(seq_str))
    sequence[-1] = seq_str
                        
    if protocol == 'Dir':
        if 'NRM' in jr6_raw.loc[jr6_raw.index[i],'treat']:
            method_code.append('LT-AF-Z') 
            experiment.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-DIR-AF') 
            measurement.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-DIR-AF'+'-'+str(i)) 
            treat_temp.append(273)
            treat_ac_field.append(0)
            #check for double NRM steps
            if i < len(jr6_raw):
                if 'NRM' in jr6_raw.loc[jr6_raw.index[i+1],'treat']:
                    for j in range(i,len(jr6_raw)):
                        if not 'NRM' in jr6_raw.loc[jr6_raw.index[j],'treat']:
                            if 'TD' in jr6_raw.loc[jr6_raw.index[j],'treat']:
                                method_code[-1] = 'LT-T-Z'
                                experiment[-1] = str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-DIR-T'
                                measurement[-1] = str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-DIR-T'+'-'+str(i)
                                break
                            elif 'AD' in jr6_raw.loc[jr6_raw.index[j],'treat']:
                                break
        if 'AD' in jr6_raw.loc[jr6_raw.index[i],'treat']: 
            method_code.append('LT-AF-Z')
            experiment.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-DIR-AF')
            measurement.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-DIR-AF'+'-'+str(i))
            treat_temp.append(273)
            treat_ac_field.append(float(jr6_raw.loc[jr6_raw.index[i],'treat'].strip('AD'))*10**-3)
        elif 'TD' in jr6_raw.loc[jr6_raw.index[i],'treat']: 
            method_code.append('LT-T-Z')
            experiment.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-DIR-T')
            measurement.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-DIR-T'+'-'+str(i))
            treat_temp.append(float(jr6_raw.loc[jr6_raw.index[i],'treat'].strip('TD'))+273)
            treat_ac_field.append(0)
            if 'NRM' in jr6_raw.loc[jr6_raw.index[i-1],'treat']:
                method_code[-2] = 'LT-T-Z'
                experiment[-2] = str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-DIR-T'
                measurement[-2] = str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-DIR-T'+'-'+str(i)
    elif protocol == 'PTSD':
        if 'NRM' in jr6_raw.loc[jr6_raw.index[i],'treat']: 
            method_code.append('LT-NO:LP-DIR-AF:LP-PI-TRM:LP-PI-ARM:LP-PI-ALT-AFARM')
            experiment.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-PI-TRM:LP-PI-ARM:LP-PI-ALT-AFARM')
            measurement.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-PI-TRM:LP-PI-ARM:LP-PI-ALT-AFARM'+'-'+str(i))
            treat_temp.append(273)
            treat_ac_field.append(0)
            treat_dc_field.append(str(0*DC_field*10**-6))
            treat_dc_field_phi.append(str(DC_phi))
            treat_dc_field_theta.append(str(DC_theta))
        if 'AD' in jr6_raw.loc[jr6_raw.index[i],'treat']: 
            if not jr6_raw.loc[jr6_raw.index[i],'treat'].endswith('.01'):
                method_code.append('LT-AF-Z:LP-DIR-AF:LP-PI-TRM:LP-PI-ARM:LP-PI-ALT-AFARM')
                treat_temp.append(273)
                treat_ac_field.append(float(jr6_raw.loc[jr6_raw.index[i],'treat'].strip('AD'))*10**-3)
                experiment.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-PI-TRM:LP-PI-ARM:LP-PI-ALT-AFARM')
                measurement.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-PI-TRM:LP-PI-ARM:LP-PI-ALT-AFARM'+'-'+str(i))
                treat_dc_field.append(str(0*DC_field*10**-6))
                treat_dc_field_phi.append(str(DC_phi))
                treat_dc_field_theta.append(str(DC_theta))
            else:
                method_code.append('LT-AF-Z:LP-ARM-AFD:LP-PI-TRM:LP-PI-ARM:LP-PI-ALT-AFARM')
                treat_temp.append(273)
                treat_ac_field.append(float(jr6_raw.loc[jr6_raw.index[i],'treat'][:-1].strip('AD'))*10**-3)
                experiment.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-PI-TRM:LP-PI-ARM:LP-PI-ALT-AFARM')
                measurement.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-PI-TRM:LP-PI-ARM:LP-PI-ALT-AFARM'+'-'+str(i))
                treat_dc_field.append(str(0*DC_field*10**-6))
                treat_dc_field_phi.append(str(DC_phi))
                treat_dc_field_theta.append(str(DC_theta))

        elif 'TD' in jr6_raw.loc[jr6_raw.index[i],'treat']: 
            method_code.append('LT-T-I:LP-TRM-AFD:LP-PI-TRM:LP-PI-ARM:LP-PI-ALT-AFARM')
            treat_temp.append(float(jr6_raw.loc[jr6_raw.index[i],'treat'].strip('TD'))+273)
            treat_ac_field.append(0)
            experiment.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-PI-TRM:LP-PI-ARM:LP-PI-ALT-AFARM')
            measurement.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-PI-TRM:LP-PI-ARM:LP-PI-ALT-AFARM'+'-'+str(i))
            treat_dc_field.append(str(DC_field*10**-6))
            treat_dc_field_phi.append(str(DC_phi))
            treat_dc_field_theta.append(str(DC_theta))
        elif 'ARM' in jr6_raw.loc[jr6_raw.index[i],'treat']: 
            method_code.append('LT-AF-I:LP-ARM-AFD:LP-PI-TRM:LP-PI-ARM:LP-PI-ALT-AFARM')
            treat_temp.append(273)
            treat_ac_field.append(float(jr6_raw.loc[jr6_raw.index[i],'treat'].strip('ARM'))*10**-3)
            experiment.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-PI-TRM:LP-PI-ARM:LP-PI-ALT-AFARM')
            measurement.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-PI-TRM:LP-PI-ARM:LP-PI-ALT-AFARM'+'-'+str(i))
            treat_dc_field.append(str(0*DC_field*10**-6))
            treat_dc_field_phi.append(str(DC_phi))
            treat_dc_field_theta.append(str(DC_theta))
    elif protocol == 'IZZI':
        #print(jr6_raw.loc[jr6_raw.index[i],'treat'])
        if 'NRM' in jr6_raw.loc[jr6_raw.index[i],'treat']: 
            method_code.append('LT-NO:LP-PI-TRM:LP-PI-ALT-PTRM:LP-PI-BT-IZZI')
            experiment.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-PI-TRM_LP-PI-ALT-PTRM_LP-PI-BT-IZZI')
            measurement.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-PI-TRM_LP-PI-ALT-PTRM_LP-PI-BT-IZZI'+'-'+str(i))
            treat_temp.append(273)
            treat_ac_field.append(0)
            treat_dc_field.append(str(0*DC_field*10**-6))
            treat_dc_field_phi.append(str(DC_phi))
            treat_dc_field_theta.append(str(DC_theta))
        elif str(jr6_raw.loc[jr6_raw.index[i],'treat'])[-2:] == '.0': # Z
            treat_temp.append(str(float(str(jr6_raw.loc[jr6_raw.index[i],'treat'].strip('TD'))[:-2])+273)) # K
            treat_dc_field.append(str(0*DC_field*10**-6))
            treat_ac_field.append(str(0)) # T
            treat_dc_field_phi.append(str(DC_phi))
            treat_dc_field_theta.append(str(DC_theta))
            if 'LT-NO' in method_code[-1] or 'LT-PTRM-I' in method_code[-1] \
                or ('LT-T-Z' in method_code[-1] and 'LP-PI-TRM-IZ' in method_code[-1]):
                    method_code.append('LT-T-Z:LP-PI-TRM-ZI:LP-PI-TRM:LP-PI-ALT-PTRM:LP-PI-BT-IZZI')
            else:
                    method_code.append('LT-T-Z:LP-PI-TRM-IZ:LP-PI-TRM:LP-PI-ALT-PTRM:LP-PI-BT-IZZI')
            experiment.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-PI-TRM_LP-PI-ALT-PTRM_LP-PI-BT-IZZI')
            measurement.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-PI-TRM_LP-PI-ALT-PTRM_LP-PI-BT-IZZI'+'-'+str(seq_str))
        elif str(jr6_raw.loc[jr6_raw.index[i],'treat'])[-2:] == '.1': # I
            treat_temp.append(str(float(str(jr6_raw.loc[jr6_raw.index[i],'treat'].strip('TD'))[:-2])+273)) # K
            treat_dc_field.append(str(DC_field*10**-6))
            treat_ac_field.append(str(0)) # T
            treat_dc_field_phi.append(str(DC_phi))
            treat_dc_field_theta.append(str(DC_theta))
            if 'LT-PTRM-I' in method_code[-1] \
                or ('LT-T-Z' in method_code[-1] and 'LP-PI-TRM-ZI' in method_code[-1]):
                method_code.append('LT-T-I:LP-PI-TRM-ZI:LP-PI-TRM:LP-PI-ALT-PTRM:LP-PI-BT-IZZI')
            else:
                method_code.append('LT-T-I:LP-PI-TRM-IZ:LP-PI-TRM:LP-PI-ALT-PTRM:LP-PI-BT-IZZI')
            experiment.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-PI-TRM_LP-PI-ALT-PTRM_LP-PI-BT-IZZI')
            measurement.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-PI-TRM_LP-PI-ALT-PTRM_LP-PI-BT-IZZI'+'-'+str(seq_str))
        elif str(jr6_raw.loc[jr6_raw.index[i],'treat'])[-2:] == '.2': # P
            treat_temp.append(str(float(str(jr6_raw.loc[jr6_raw.index[i],'treat'].strip('TD'))[:-2])+273)) # K
            treat_dc_field.append(str(DC_field*10**-6))
            treat_ac_field.append(str(0)) # T
            treat_dc_field_phi.append(str(DC_phi))
            treat_dc_field_theta.append(str(DC_theta))
            method_code.append('LT-PTRM-I:LP-PI-TRM:LP-PI-ALT-PTRM:LP-PI-BT-IZZI')
            experiment.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-PI-TRM_LP-PI-ALT-PTRM_LP-PI-BT-IZZI')
            measurement.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-PI-TRM_LP-PI-ALT-PTRM_LP-PI-BT-IZZI'+'-'+str(seq_str))
        elif str(jr6_raw.loc[jr6_raw.index[i],'treat'])[-2:] == '.3': # T
            treat_temp.append(str(float(str(jr6_raw.loc[jr6_raw.index[i],'treat'].strip('TD'))[:-2])+273)) # K
            treat_dc_field.append(str(0*DC_field*10**-6))
            treat_ac_field.append(str(0)) # T
            treat_dc_field_phi.append(str(DC_phi))
            treat_dc_field_theta.append(str(DC_theta))
            method_code.append('LT-PTRM-MD:LP-PI-TRM:LP-PI-ALT-PTRM:LP-PI-BT-IZZI')
            experiment.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-PI-TRM_LP-PI-ALT-PTRM_LP-PI-BT-IZZI')
            measurement.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-PI-TRM_LP-PI-ALT-PTRM_LP-PI-BT-IZZI'+'-'+str(seq_str))
        elif str(jr6_raw.loc[jr6_raw.index[i],'treat'])[-2:] == '.4': # A
            treat_temp.append(str(float(str(jr6_raw.loc[jr6_raw.index[i],'treat'].strip('TD'))[:-2])+273)) # K
            treat_dc_field.append(str(0*DC_field*10**-6))
            treat_ac_field.append(str(0)) # T
            treat_dc_field_phi.append(str(DC_phi))
            treat_dc_field_theta.append(str(DC_theta))
            method_code.append('LT-PTRM-AC:LP-PI-TRM:LP-PI-ALT-PTRM:LP-PI-BT-IZZI')
            experiment.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-PI-TRM_LP-PI-ALT-PTRM_LP-PI-BT-IZZI')
            measurement.append(str(jr6_raw.loc[jr6_raw.index[i],'specimen'])+'_'+'LP-PI-TRM_LP-PI-ALT-PTRM_LP-PI-BT-IZZI'+'-'+str(seq_str))
    else:
        print('protocol has not been added yet. contact whoever is responsible for this notebook')
    lastname = jr6_raw.loc[jr6_raw.index[i],'specimen']    
jr6_raw['site'] = site
jr6_raw['sample'] = sample
jr6_raw['method_codes'] = method_code
jr6_raw['experiment'] = experiment
jr6_raw['measurement'] = measurement
jr6_raw['sequence'] = sequence
jr6_raw['treat_step_num'] = treat_step_num
jr6_raw['treat_temp'] = treat_temp
jr6_raw['treat_ac_field'] = treat_ac_field
if not protocol == 'Dir':
    jr6_raw['treat_dc_field'] = treat_dc_field
    jr6_raw['treat_dc_field_phi'] = treat_dc_field_phi
    jr6_raw['treat_dc_field_theta'] = treat_dc_field_theta
                                                   
sites_new = list(set(jr6_raw['site']))
for i in range(len(sites_new)):
    if not sites_new[i] in sites_dict['site']:
        sites_dict['site'].append(sites_new[i])
for i in range(len(sites_dict['site'])):
    temp_df = jr6_raw[jr6_raw['site'] == sites_dict['site'][i]]
    if not len(temp_df) == 0:
        sites_dict['method_codes'].append(':'.join(list(set(temp_df['method_codes'].astype(str)))))
        sites_dict['citations'].append(':'.join(list(set(temp_df['citations'].astype(str)))))
        sites_dict['method_codes'][-1] = sites_dict['method_codes'][-1].replace('LT-T-Z','LP-DIR-T').replace('LT-AF-Z','LP-DIR-AF')
samples_new = list(set(jr6_raw['sample']))
for i in range(len(samples_new)):
    if not samples_new[i] in samples_dict:
        samples_dict['sample'].append(samples_new[i])
for i in range(len(samples_dict['sample'])):
    temp_df = jr6_raw[jr6_raw['sample'] == samples_dict['sample'][i]]
    if not len(temp_df) == 0:
        for key in samples_dict.keys():
            if not key == 'sample':
                samples_dict[key].append(':'.join(list(set(temp_df[key].astype(str)))))
            if key == 'method_codes':
                samples_dict['method_codes'][-1] = samples_dict['method_codes'][-1].replace('LT-T-Z','LP-DIR-T').replace('LT-AF-Z','LP-DIR-AF')
specimens_new = list(set(jr6_raw['specimen']))
for i in range(len(specimens_new)):
    if not specimens_new[i] in specimens_dict:
        specimens_dict['specimen'].append(specimens_new[i])
for i in range(len(specimens_dict['specimen'])):
    temp_df = jr6_raw[jr6_raw['specimen'] == specimens_dict['specimen'][i]]
    if not len(temp_df) == 0:
        for key in specimens_dict.keys():
            if key == 'experiments':
                specimens_dict[key].append(':'.join(list(set(temp_df['experiment'].astype(str)))))
            elif not key == 'specimen':
                specimens_dict[key].append(':'.join(list(set(temp_df[key].astype(str)))))    
            if key == 'method_codes':
                specimens_dict['method_codes'][-1] = specimens_dict['method_codes'][-1].replace('LT-T-Z','LP-DIR-T').replace('LT-AF-Z','LP-DIR-AF')

for key in measurement_dict.keys():
    measurement_dict[key] = jr6_raw[key].values
print('Done!')

Done!


In [4]:
# save files for pmagpy
savepath = r'G:\My Drive\__Postdoc\Data\FSU\Panther_Creek'

if not savepath.endswith('/'): savepath += '/'
with open(savepath+'sites.txt', 'w') as sites_file:
    sites_file.write('tab\tsites\n')
sites_df = pd.DataFrame(sites_dict)
sites_df.to_csv(savepath+'sites.txt',mode='a',index=False,sep='\t')
    
with open(savepath+'samples.txt', 'w') as samples_file:
    samples_file.write('tab\tsamples\n')
samples_df = pd.DataFrame(samples_dict)
samples_df.to_csv(savepath+'samples.txt',mode='a',index=False,sep='\t')

with open(savepath+'specimens.txt', 'w') as specimens_file:
    specimens_file.write('tab\tspecimens\n')
specimens_df = pd.DataFrame(specimens_dict)
specimens_df.to_csv(savepath+'specimens.txt',mode='a',index=False,sep='\t')

with open(savepath+'measurements.txt', 'w') as measurements_file:
    measurements_file.write('tab\tmeasurements\n')
measurement_df = pd.DataFrame(measurement_dict)
measurement_df = measurement_df[(~measurement_df['specimen'].isna())]
#measurement_df = measurement_df.sort_values(by=['specimen','sequence'])
measurement_df.to_csv(savepath+'measurements.txt',mode='a',index=False,sep='\t')

print('Files for pmagpy GUI saved to', os.getcwd())

Files for pmagpy GUI saved to G:\My Drive\__Postdoc\Misc\Side_quests\UF2MagIC
